# LLaVA Fine-tuning for Fall Hazard Detection

This notebook fine-tunes LLaVA 7B on the PHELE dataset for fall hazard detection using **LoRA** (Low-Rank Adaptation) with **Unsloth** for efficient training.

**Requirements:**
- Google Colab with GPU (T4 or better)
- PHELE training data (generated by `prepare_training_data.py`)
- Google Drive for data storage

**Estimated Training Time:** 2-4 hours on T4 GPU

---

**Dissertation Project:** Assessing In-Home Safety Risks for Older Adults Using Generative Models

**Author:** Summen Zahid (B00996747)

**Supervisor:** Mark Donnelly

## 1. Setup Environment

First, install the required packages. This uses Unsloth for efficient LoRA training.

In [ ]:
%%capture
# Install Unsloth for efficient fine-tuning
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
# Check GPU
!nvidia-smi

## 2. Mount Google Drive & Load Data

Upload your training data to Google Drive first, then mount it here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Configure paths - UPDATE THESE TO MATCH YOUR DRIVE STRUCTURE
import os

# Path to your training data in Google Drive
DRIVE_PATH = "/content/drive/MyDrive/Dissertation"
DATA_PATH = f"{DRIVE_PATH}/training_data"
OUTPUT_PATH = f"{DRIVE_PATH}/models/llava_finetuned"

# Create output directory
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Check if training data exists
train_file = f"{DATA_PATH}/train.json"
if os.path.exists(train_file):
    print(f"Training data found: {train_file}")
else:
    print(f"ERROR: Training data not found at {train_file}")
    print("Please upload your training data to Google Drive first.")
    print("\nExpected structure:")
    print(f"  {DATA_PATH}/")
    print("    train.json")
    print("    val.json")
    print("    images/train/")
    print("    images/val/")

## 3. Load LLaVA Model with LoRA

We load LLaVA-1.5-7B with 4-bit quantization and apply LoRA adapters for efficient fine-tuning.

In [ ]:
from unsloth import FastVisionModel
import torch

# Load model with 4-bit quantization
model, tokenizer = FastVisionModel.from_pretrained(
    model_name="unsloth/llava-v1.6-mistral-7b-hf-bnb-4bit",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

print("Model loaded successfully!")

In [ ]:
# Apply LoRA adapters
model = FastVisionModel.get_peft_model(
    model,
    r=16,                       # LoRA rank (higher = more parameters)
    lora_alpha=16,              # LoRA alpha
    lora_dropout=0.05,          # Dropout for regularization
    bias="none",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

print("LoRA adapters applied!")
print(f"Trainable parameters: {model.print_trainable_parameters()}")

## 4. Prepare Dataset

Load and format the PHELE training data for fine-tuning.

In [ ]:
import json
from PIL import Image
from datasets import Dataset

def load_training_data(json_path, image_base_path):
    """Load training data from JSON file."""
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    samples = []
    for item in data:
        # Get image path
        img_path = item.get('image', '')
        if not img_path.startswith('/'):
            img_path = os.path.join(image_base_path, img_path)
        
        # Skip if image doesn't exist
        if not os.path.exists(img_path):
            print(f"Warning: Image not found: {img_path}")
            continue
        
        # Get conversations
        conversations = item.get('conversations', [])
        if len(conversations) < 2:
            continue
        
        human_msg = conversations[0].get('value', '')
        assistant_msg = conversations[1].get('value', '')
        
        samples.append({
            'id': item.get('id', ''),
            'image_path': img_path,
            'instruction': human_msg,
            'output': assistant_msg
        })
    
    return samples

# Load training data
train_samples = load_training_data(
    f"{DATA_PATH}/train.json",
    DATA_PATH
)

print(f"Loaded {len(train_samples)} training samples")

# Preview a sample
if train_samples:
    sample = train_samples[0]
    print(f"\nSample ID: {sample['id']}")
    print(f"Image: {sample['image_path']}")
    print(f"Instruction: {sample['instruction'][:100]}...")
    print(f"Output: {sample['output'][:200]}...")

In [ ]:
def format_for_training(samples, tokenizer):
    """Format samples for LLaVA training."""
    formatted = []
    
    for sample in samples:
        try:
            # Load image
            image = Image.open(sample['image_path']).convert('RGB')
            
            # Format conversation
            conversation = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": image},
                        {"type": "text", "text": sample['instruction'].replace('<image>', '')}
                    ]
                },
                {
                    "role": "assistant",
                    "content": [
                        {"type": "text", "text": sample['output']}
                    ]
                }
            ]
            
            formatted.append({
                'id': sample['id'],
                'messages': conversation,
                'images': [image]
            })
            
        except Exception as e:
            print(f"Error processing {sample['id']}: {e}")
            continue
    
    return formatted

# Format training data
print("Formatting training data...")
formatted_data = format_for_training(train_samples[:100], tokenizer)  # Limit for testing
print(f"Formatted {len(formatted_data)} samples")

## 5. Training Configuration

Set up the training parameters for LoRA fine-tuning.

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

# Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_PATH,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=100,  # Increase for full training
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    report_to="none",
)

print("Training configuration set!")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Max steps: {training_args.max_steps}")

## 6. Train the Model

Run the fine-tuning process. This will take 2-4 hours on a T4 GPU.

In [ ]:
from unsloth import is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator

# Create trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=formatted_data,
    args=training_args,
)

print("Starting training...")
print("This will take 2-4 hours on a T4 GPU.")

In [ ]:
# Train!
trainer_stats = trainer.train()

print("\nTraining completed!")
print(f"Total steps: {trainer_stats.global_step}")
print(f"Training loss: {trainer_stats.training_loss:.4f}")

## 7. Save the Model

Save the fine-tuned LoRA adapters and optionally merge with base model.

In [ ]:
# Save LoRA adapters (small, efficient)
lora_path = f"{OUTPUT_PATH}/lora_adapters"
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)

print(f"LoRA adapters saved to: {lora_path}")

In [ ]:
# Optional: Merge LoRA with base model and save in GGUF format for Ollama
# This creates a larger file but is easier to use

SAVE_GGUF = True  # Set to True to export for Ollama

if SAVE_GGUF:
    gguf_path = f"{OUTPUT_PATH}/llava_hazard_gguf"
    
    # Save in multiple quantization levels
    model.save_pretrained_gguf(
        gguf_path,
        tokenizer,
        quantization_method="q4_k_m"  # Good balance of size/quality
    )
    
    print(f"\nGGUF model saved to: {gguf_path}")
    print("\nTo use with Ollama:")
    print("1. Download the .gguf file from Google Drive")
    print("2. Create a Modelfile with:")
    print('   FROM ./llava_hazard.gguf')
    print('   PARAMETER temperature 0.1')
    print("3. Run: ollama create llava-hazard -f Modelfile")

## 8. Test the Model

Run inference on a test image to verify the fine-tuning worked.

In [ ]:
# Load a test image
test_image_path = f"{DATA_PATH}/images/test/"  # Update with your test image

# Find first test image
test_images = [f for f in os.listdir(test_image_path) if f.endswith(('.jpg', '.png', '.jpeg'))]

if test_images:
    test_img_path = os.path.join(test_image_path, test_images[0])
    test_image = Image.open(test_img_path).convert('RGB')
    
    # Display image
    from IPython.display import display
    display(test_image.resize((400, 300)))
    
    print(f"Testing on: {test_images[0]}")
else:
    print("No test images found. Upload a test image to test the model.")

In [ ]:
# Run inference
if test_images:
    FastVisionModel.for_inference(model)
    
    prompt = """Analyze this home environment image and identify all fall hazards that could affect elderly residents.

For each hazard, provide:
1. Category
2. Specific type of hazard
3. Severity level (low, medium, high, critical)
4. Brief description"""
    
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": test_image},
                {"type": "text", "text": prompt}
            ]
        }
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.1,
        do_sample=True
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    print("\n" + "="*60)
    print("MODEL OUTPUT")
    print("="*60)
    print(response)

## 9. Next Steps

After training:

1. **Download the model** from Google Drive
2. **For Ollama usage:**
   - Download the GGUF file
   - Create a Modelfile:
     ```
     FROM ./llava_hazard.gguf
     PARAMETER temperature 0.1
     SYSTEM "You are an expert occupational therapist..."
     ```
   - Run: `ollama create llava-hazard -f Modelfile`
   
3. **Run evaluation:**
   ```bash
   python -m src.evaluation.run_evaluation --model llava-hazard --split test
   ```

4. **Compare with baseline:**
   ```bash
   python -m src.evaluation.run_evaluation --model llava llava-hazard --split test
   ```

In [ ]:
# Print final summary
print("\n" + "="*60)
print("FINE-TUNING COMPLETE")
print("="*60)
print(f"\nModel saved to: {OUTPUT_PATH}")
print(f"\nFiles created:")
for f in os.listdir(OUTPUT_PATH):
    print(f"  - {f}")
print("\nDownload these files from Google Drive to use locally.")